# ProtoPregNet vs. all baselines — statistical significance (true paired bootstrap)

Loads held-out test predictions exported by:
- `protopregnet_raw_embedding_baselines.ipynb` → raw-embeddings LogReg / DT / RF
- `protopregnet_non_interp_baseline.ipynb` → adapter+linear (no prototype layer)
- `fusion.ipynb` → ablation of ProtoPregNet
- `tabular_ehr_baselines_full_record.ipynb` → tabular Joint RF / DT / RF / LogReg

Each baseline is **aligned by `person_id`** to ProtoPregNet's fusion test population before paired resampling. Overlap size is printed per baseline (tabular uses a different split, so only the intersection is paired).

In [0]:
# !pip install torch

In [0]:
import os, pickle
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    brier_score_loss, balanced_accuracy_score,
)

CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]

EXPORT_DIR = "/path/to/exports"
REF_PATH = os.path.join(EXPORT_DIR, "full_analysis_export.pkl")
NON_INTERP_PATH = os.path.join(EXPORT_DIR, "protopregnet_non_interp_test_predictions.pkl")
ABLATION_PATH = os.path.join(EXPORT_DIR, "ablation_test_predictions.pkl")
TABULAR_PATH = os.path.join(EXPORT_DIR, "tabular_ehr_test_predictions.pkl")
RAW_PATH = os.path.join(EXPORT_DIR, "protopregnet_raw_embedding_test_predictions.pkl")

N_BOOTSTRAP = 1000
BOOTSTRAP_CONFIDENCE = 0.95
BOOTSTRAP_SEED = 42
SIGNIFICANCE_METRICS = ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro"]


In [0]:
def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    results["auroc_macro"] = macro("auroc")
    results["auprc_macro"] = macro("auprc")
    results["f1_macro"] = macro("f1")
    results["bacc_macro"] = macro("bacc")
    results["brier_macro"] = macro("brier")
    return results


def load_pred_payload(path):
    with open(path, "rb") as f:
        payload = pickle.load(f)
    payload["person_id"] = np.asarray(payload["person_id"]).astype(str)
    payload["Y_true"] = np.asarray(payload["Y_true"])
    payload["predictions"] = {k: np.asarray(v) for k, v in payload["predictions"].items()}
    return payload


def align_to_reference(ref_pids, other_pids, other_Y, label="baseline"):
    """Return (ref_idx, other_idx, Y_other) on the person_id intersection,
    ordered to match ProtoPregNet's ref_pids sequence."""
    lookup = {pid: i for i, pid in enumerate(other_pids)}
    common = [pid for pid in ref_pids if pid in lookup]
    if len(common) == 0:
        raise ValueError(f"No overlapping person_ids for {label}")
    other_idx = np.array([lookup[pid] for pid in common])
    ref_lookup = {pid: i for i, pid in enumerate(ref_pids)}
    ref_idx = np.array([ref_lookup[pid] for pid in common])
    Y_other = other_Y[other_idx]
    n_miss = len(ref_pids) - len(common)
    print(f"  {label}: aligned n={len(common)} / ref n={len(ref_pids)}"
          + (f" ({n_miss} ref patients not in this baseline's test set)" if n_miss else ""))
    return ref_idx, other_idx, Y_other


def paired_bootstrap_significance(Y_true, p_a, p_b, metric_key="auroc_macro",
                                  n_bootstrap=N_BOOTSTRAP, confidence=BOOTSTRAP_CONFIDENCE,
                                  seed=BOOTSTRAP_SEED, label_a="ProtoPregNet", label_b="baseline"):
    """p_a is the reference model (ProtoPregNet); p_b the baseline.
    diff = metric(p_a) - metric(p_b), computed on the SAME resample each
    iteration (p_a, p_b, Y_true must be row-aligned)."""
    rng = np.random.RandomState(seed)
    n = Y_true.shape[0]
    diffs = []
    n_skipped = 0
    lower_is_better = metric_key.startswith("brier")
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yt_r = Y_true[idx]
        degenerate = any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(CLASS_NAMES)))
        if degenerate:
            n_skipped += 1
            continue
        m_a = _compute_metric_suite(yt_r, p_a[idx])[metric_key]
        m_b = _compute_metric_suite(yt_r, p_b[idx])[metric_key]
        if np.isnan(m_a) or np.isnan(m_b):
            continue
        diffs.append(m_a - m_b)

    diffs = np.array(diffs)
    mean_diff = float(diffs.mean())
    lo, hi = np.percentile(diffs, [(1 - confidence) / 2 * 100, (1 + confidence) / 2 * 100])
    if lower_is_better:
        p_value = float(np.mean(diffs >= 0))
    else:
        p_value = float(np.mean(diffs <= 0))

    sig = "(significant, alpha=0.05)" if (lo > 0 or hi < 0) else "(not significant)"
    print(f"  {label_a} - {label_b}  [{metric_key}]: mean diff {mean_diff:+.4f}  "
          f"(95% CI [{lo:+.4f}, {hi:+.4f}])  one-sided p={p_value:.4f}  {sig}")
    if n_skipped > 0:
        print(f"    ({n_skipped}/{n_bootstrap} resamples skipped -- degenerate label in that draw)")
    return mean_diff, float(lo), float(hi), p_value


## load exports + align baselines to ProtoPregNet test population

In [0]:
def load_ref_from_full_export(path):
    """full_analysis_export.pkl stores pids_test / Y_fusion_test / p_test; convert to the standard
    (person_id, Y_true, predictions) layout so the rest of the notebook is unchanged."""
    with open(path, "rb") as f:
        exp = pickle.load(f)
    return {
        "person_id": np.asarray(exp["pids_test"]).astype(str),
        "Y_true": np.asarray(exp["Y_fusion_test"]),
        "predictions": {"ProtoPregNet": np.asarray(exp["p_test"])},
    }


ref = load_ref_from_full_export(REF_PATH)
ref_pids = ref["person_id"]
Y_ref = ref["Y_true"]
p_proto = ref["predictions"]["ProtoPregNet"]
assert len(ref_pids) == len(Y_ref) == len(p_proto)
assert p_proto.shape == Y_ref.shape, f"ProtoPregNet probs {p_proto.shape} vs Y {Y_ref.shape}"

# the full export holds ProtoPregNet only; the raw-embed baselines (previous reference export) are no longer in it
aligned_baselines = {name: p for name, p in ref["predictions"].items() if name != "ProtoPregNet"}
print(f"Reference: ProtoPregNet (n={len(ref_pids)}) + {len(aligned_baselines)} already-aligned baselines")

# external exports: aligned by person_id (tabular may hold a different / larger test set -> intersection only)
external_paths = [RAW_PATH, NON_INTERP_PATH, TABULAR_PATH, ABLATION_PATH] 

SKIP_NAMES = {"Full Fusion"}   # the ablation export carries a copy of the full model; testing it against itself is meaningless
baseline_meta = {}  # name -> dict with p_b, Y_pair, p_a_pair (possibly subset)

for name, p_b in aligned_baselines.items():
    baseline_meta[name] = {"Y_true": Y_ref, "p_a": p_proto, "p_b": p_b, "n": len(ref_pids)}

print("\nAligning external baseline exports:")
for path in external_paths:
    if not os.path.exists(path):
        print(f"  SKIP missing: {path}")
        continue
    payload = load_pred_payload(path)
    for name, p_other in payload["predictions"].items():
        if name in SKIP_NAMES:
            continue
        ref_idx, other_idx, Y_other = align_to_reference(
            ref_pids, payload["person_id"], payload["Y_true"], label=name
        )
        Y_pair = Y_ref[ref_idx]
        # prefer ProtoPregNet labels; warn if disagreement
        if not np.array_equal(Y_pair.astype(int), Y_other.astype(int)):
            n_disagree = int((Y_pair.astype(int) != Y_other.astype(int)).any(axis=1).sum())
            print(f"    WARNING: {n_disagree} patients have label mismatch vs ProtoPregNet; using ProtoPregNet Y_true")
        baseline_meta[name] = {
            "Y_true": Y_pair,
            "p_a": p_proto[ref_idx],
            "p_b": p_other[other_idx],
            "n": len(ref_idx),
        }

print(f"\nBaselines ready for paired tests ({len(baseline_meta)}):")
for name, m in baseline_meta.items():
    print(f"  {name:40s} n={m['n']}")


## statistical significance vs. ProtoPregNet (true paired bootstrap)

In [0]:
def bootstrap_raw_metrics(Y_true, p_a, p_b, metric_keys, n_bootstrap=N_BOOTSTRAP, confidence=BOOTSTRAP_CONFIDENCE, seed=BOOTSTRAP_SEED):
    """Point estimate (full aligned set) + percentile bootstrap CI of each metric, for both models, in ONE pass.
    Same seed and draw order as paired_bootstrap_significance, so these CIs come from the very same resamples as the paired difference."""
    rng = np.random.RandomState(seed)
    n = Y_true.shape[0]
    point_a, point_b = _compute_metric_suite(Y_true, p_a), _compute_metric_suite(Y_true, p_b)
    draws = {k: ([], []) for k in metric_keys}
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yt_r = Y_true[idx]
        if any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(CLASS_NAMES))):
            continue                                   # undefined metrics in this draw -> skipped, as in the paired test
        m_a, m_b = _compute_metric_suite(yt_r, p_a[idx]), _compute_metric_suite(yt_r, p_b[idx])
        for k in metric_keys:
            draws[k][0].append(m_a[k]); draws[k][1].append(m_b[k])
    q = [(1 - confidence) / 2 * 100, (1 + confidence) / 2 * 100]
    out = {}
    for k in metric_keys:
        res = []
        for point, arr in ((point_a[k], draws[k][0]), (point_b[k], draws[k][1])):
            arr = np.asarray(arr, dtype=float); arr = arr[~np.isnan(arr)]
            lo, hi = np.percentile(arr, q) if len(arr) else (float("nan"), float("nan"))
            res.append((float(point), float(lo), float(hi)))
        out[k] = {"a": res[0], "b": res[1]}
    return out


sig_results = {}
raw_results = {}     # name -> metric -> {"a": (estimate, ci_lo, ci_hi) for ProtoPregNet, "b": same for the baseline}
for name, m in baseline_meta.items():
    print(f"\n=== ProtoPregNet vs. {name} (n={m['n']}) ===")
    sig_results[name] = {}
    raw_results[name] = bootstrap_raw_metrics(m["Y_true"], m["p_a"], m["p_b"], SIGNIFICANCE_METRICS)
    for metric_key in SIGNIFICANCE_METRICS:
        (ea, la, ha), (eb, lb, hb) = raw_results[name][metric_key]["a"], raw_results[name][metric_key]["b"]
        print(f"  [{metric_key}]  ProtoPregNet {ea:.4f} (95% CI [{la:.4f}, {ha:.4f}])  |  {name} {eb:.4f} (95% CI [{lb:.4f}, {hb:.4f}])")
        sig_results[name][metric_key] = paired_bootstrap_significance(
            m["Y_true"], m["p_a"], m["p_b"], metric_key=metric_key,
            label_a="ProtoPregNet", label_b=name,
        )